# LLM Power Law - Benchmarking Framework on Google Colab

This notebook helps you run LLM benchmarking experiments on Google Colab with free GPU access.

**Hardware:** Colab provides ~15GB VRAM (T4 GPU) - enough for models up to 13B with 4-bit quantization!

## Quick Start

1. **Enable GPU**: Runtime → Change runtime type → GPU (T4)
2. **Run all cells** in order
3. **Monitor progress** with built-in progress bars
4. **Download results** from the Files panel (left sidebar)

## 1. Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\n✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# Clone or update repository
import os

# Always work from /content directory in Colab
base_dir = "/content"
repo_url = "https://github.com/YOUR_USERNAME/LLMPowerLaw.git"
repo_name = "LLMPowerLaw"
repo_path = os.path.join(base_dir, repo_name)

# Change to base directory first
os.chdir(base_dir)
print(f"📂 Working directory: {os.getcwd()}")

if os.path.exists(repo_path):
    print(f"📂 Repository already exists. Updating to latest version...")
    os.chdir(repo_path)
    
    # Discard any local changes (config files modified by notebook cells)
    !git reset --hard HEAD
    !git clean -fd
    
    # Pull latest changes
    !git pull origin main
    print("✅ Repository updated to latest version!")
else:
    print(f"📥 Cloning repository for first time...")
    !git clone {repo_url}
    os.chdir(repo_path)
    print("✅ Repository cloned!")

print(f"✅ Current directory: {os.getcwd()}")
print(f"✅ Files: {os.listdir('.')[:10]}")  # Show first 10 files to verify

In [ ]:
# Install dependencies
print("📦 Installing core dependencies...")
!pip install -q torch transformers accelerate datasets

print("📦 Installing utilities...")
!pip install -q tqdm pyyaml python-dotenv pandas numpy scikit-learn

print("📦 Installing quantization support (4-bit models)...")
!pip install -q bitsandbytes

print("📦 Installing optional packages...")
!pip install -q sentencepiece --only-binary :all:

print("\n✅ Installation complete!")

## 2. Configure Models & Datasets

Colab's free T4 GPU (~15GB VRAM) can run:
- ✅ 7B models with 4-bit quantization (~4GB VRAM)
- ✅ 13B models with 4-bit quantization (~7GB VRAM)
- ✅ Multiple small models (2-3B)

**Pre-configured options below** - just uncomment what you want to test!

### Mode A: Triplet Mode (Recommended) ✅

Define explicit experiments in `experiments.yaml`. Each experiment specifies model + dataset + prompting technique.
This lets you write model-specific prompt templates manually!

In [ ]:
# TRIPLET MODE: Configure explicit experiments
# Each experiment is a model-dataset-template triplet

import yaml

# Read experiments config
with open('config/experiments.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

print("📋 Available experiments:")
for exp in config['experiments']:
    status = "✅" if exp['enabled'] else "⬜"
    print(f"{status} [{exp['id']}] {exp['model']} → {exp['dataset']} → {exp['prompting_technique']}")

print("\n" + "="*70)

# STEP 1: Disable ALL experiments first
for exp in config['experiments']:
    exp['enabled'] = False

# STEP 2: Enable specific experiments you want to run
experiments_to_enable = [
    'phi3_custom_simple',      # Phi-3 on custom sentiment (fast test)
    'gemma_custom_simple',     # Gemma on custom sentiment  
    'tinyllama_custom_simple', # TinyLlama on custom sentiment
    # 'phi3_sst2_simple',      # Uncomment for SST-2 dataset
    # 'llama2_7b_qqp',         # Uncomment for Llama-2 7B on QQP
]

# Enable selected experiments
print("\n🔧 Enabling experiments:")
for exp in config['experiments']:
    if exp['id'] in experiments_to_enable:
        exp['enabled'] = True
        print(f"✅ [{exp['id']}] {exp['model']} → {exp['dataset']} → {exp['prompting_technique']}")

# Save updated config
with open('config/experiments.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print(f"\n✅ Triplet mode configured! ({len(experiments_to_enable)} experiments)")
print("💡 Define custom prompting templates in config/prompting_techniques.yaml")
print("💡 Skip 'Mode B' cells below and jump to 'Run Benchmark'!")

### Mode A+: Batch Enable by Model × Dataset × Technique

Instead of listing individual experiment IDs, select **sets** of models, datasets, and techniques.
Every experiment whose `(model, dataset, technique)` matches the cross-product is enabled.

The `technique` filter matches the **base technique name** embedded in the `prompting_technique` field
(e.g. `zero_shot` matches `sst2_zero_shot_smollm`).

Edit the three lists below and run the cell.

---

**Available keys** (use `["*"]` for all, or glob patterns like `"qwen*"`):

**MODELS** (sorted by param count):
| Key | Model | Params |
|-----|-------|--------|
| `smollm-135m` | SmolLM-135M-Instruct | 0.135B |
| `smollm-360m` | SmolLM-360M-Instruct | 0.36B |
| `qwen2.5-0.5b` | Qwen2.5-0.5B-Instruct | 0.5B |
| `tinyllama-test` | TinyLlama-1.1B-Chat | 1.1B |
| `qwen2.5-1.5b` | Qwen2.5-1.5B-Instruct | 1.5B |
| `smollm-1.7b` | SmolLM-1.7B-Instruct | 1.7B |
| `qwen2.5-3b` | Qwen2.5-3B-Instruct | 3B |
| `phi-3-mini-4bit` | Phi-3-mini-4k-instruct | 3.8B |
| `mistral-7b-instruct` | Mistral-7B-Instruct-v0.3 | 7.2B |
| `qwen2.5-7b` | Qwen2.5-7B-Instruct | 7.6B |

**DATASETS:**
`sst2` · `arc_challenge` · `hellaswag` · `gsm8k` · `mmlu`

**TECHNIQUES:**
`zero_shot` · `few_shot` · `chain_of_thought` · `few_shot_cot` · `role_expert`

In [ ]:
# ============================================================
# BATCH ENABLE: select models, datasets, and techniques
# All matching (model × dataset × technique) combos are enabled.
# Use "*" or "all" to match everything in that dimension.
# ============================================================
import yaml, fnmatch

# ── Edit these three lists ──────────────────────────────────
MODELS     = ["smollm-135m", "smollm-360m"]   # or ["*"] for all
DATASETS   = ["sst2"]                          # or ["*"] for all
TECHNIQUES = ["zero_shot", "few_shot"]         # or ["*"] for all
# ─────────────────────────────────────────────────────────────

# Known base technique names (the part between dataset_ and _family)
BASE_TECHNIQUES = [
    "zero_shot", "few_shot", "chain_of_thought", "few_shot_cot", "role_expert",
]

def _matches(value, patterns):
    """Check if value matches any pattern (supports *, all, fnmatch globs)."""
    for p in patterns:
        if p in ("*", "all") or fnmatch.fnmatch(value, p):
            return True
    return False

def _extract_base_technique(prompting_technique: str) -> str:
    """Extract the base technique name from a generated template name.
    e.g. 'sst2_zero_shot_smollm' → 'zero_shot'
         'arc_challenge_few_shot_cot_qwen2_5' → 'few_shot_cot'
         'classification_simple_phi_3_mini' → 'classification_simple_phi_3_mini' (unchanged)
    """
    for bt in sorted(BASE_TECHNIQUES, key=len, reverse=True):
        if f"_{bt}_" in prompting_technique or prompting_technique.endswith(f"_{bt}"):
            return bt
    return prompting_technique  # fallback: use the raw name

# Load experiments
with open("config/experiments.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# Disable all, then selectively enable
enabled_count = 0
for exp in config["experiments"]:
    exp["enabled"] = False
    model_ok    = _matches(exp["model"], MODELS)
    dataset_ok  = _matches(exp["dataset"], DATASETS)
    base_tech   = _extract_base_technique(exp.get("prompting_technique", ""))
    tech_ok     = _matches(base_tech, TECHNIQUES)
    if model_ok and dataset_ok and tech_ok:
        exp["enabled"] = True
        enabled_count += 1

# Save
with open("config/experiments.yaml", "w", encoding="utf-8") as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

# Summary
print(f"✅ Enabled {enabled_count} experiments  "
      f"(models={MODELS}  datasets={DATASETS}  techniques={TECHNIQUES})\n")
if enabled_count:
    print(f"{'ID':<35} {'MODEL':<22} {'DATASET':<18} {'TECHNIQUE'}")
    print("─" * 110)
    for exp in config["experiments"]:
        if exp["enabled"]:
            print(f"{exp['id']:<35} {exp['model']:<22} {exp['dataset']:<18} {exp['prompting_technique']}")
else:
    print("⚠️  No experiments matched. Check MODELS / DATASETS / TECHNIQUES above.")

### Mode B: Auto Mode (Legacy)

Enable models, datasets, and techniques separately. All combinations will run automatically.
⚠️ Use Mode A above for model-specific prompting templates!

In [ ]:
# View current configuration
!cat config/models.yaml | head -n 100

In [ ]:
# Quick configuration for Colab (15GB VRAM)
# This enables models that work well on Colab's free tier

import yaml

# Read current config
with open('config/models.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# STEP 1: Disable ALL models first
for model in config['models']:
    model['enabled'] = False

# STEP 2: Enable only recommended models for Colab
models_to_enable = [
    'tinyllama-test',      # Fast testing (1.1B)
    'gemma-2b-4bit',       # Excellent quality (2B)
    'phi-3-mini-4bit',     # Balanced (3.8B)
    # 'llama-2-7b-4bit',     # High quality (7B) - Uncomment if needed
]

# Optionally enable for full Colab power (comment out if testing)
# models_to_enable.append('llama-2-13b-4bit')  # Needs ~7GB VRAM

for model in config['models']:
    if model['name'] in models_to_enable:
        model['enabled'] = True
        print(f"✅ Enabled: {model['name']}")

# Save config
with open('config/models.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print(f"\n✅ Model configuration updated! ({len(models_to_enable)} models enabled)")

In [ ]:
# Configure datasets - start with small test
import yaml

with open('config/datasets.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# STEP 1: Disable ALL datasets first
for dataset in config['datasets']:
    dataset['enabled'] = False

# STEP 2: Enable only test dataset (5 samples - fast verification)
datasets_to_enable = [
    'custom_classification_test',  # 5 samples for quick test
    # 'sst2_test',                 # Uncomment for SST-2 test (10 samples)
]

for dataset in config['datasets']:
    if dataset['name'] in datasets_to_enable:
        dataset['enabled'] = True
        num_samples = dataset.get('num_samples', 'all')
        print(f"✅ Enabled: {dataset['name']} ({num_samples} samples)")

# Save
with open('config/datasets.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print(f"\n✅ Dataset configuration updated! ({len(datasets_to_enable)} datasets enabled)")
print("\n💡 After test succeeds, enable more datasets or increase num_samples")

In [ ]:
# Configure prompting techniques
import yaml

with open('config/prompting_techniques.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# STEP 1: Disable ALL prompting techniques first
for technique in config['prompting_techniques']:
    technique['enabled'] = False

# STEP 2: Enable only specific techniques
techniques_to_enable = [
    'classification_simple',  # Custom classification prompt
    # 'zero_shot',            # Uncomment for zero-shot baseline
    # 'few_shot',             # Uncomment for few-shot examples
]

for technique in config['prompting_techniques']:
    if technique['name'] in techniques_to_enable:
        technique['enabled'] = True
        print(f"✅ Enabled technique: {technique['name']}")

# STEP 3: Update dataset-to-technique mapping (IMPORTANT!)
# This ensures each dataset uses the correct prompting technique
if 'global_settings' not in config:
    config['global_settings'] = {}

if 'dataset_technique_mapping' not in config['global_settings']:
    config['global_settings']['dataset_technique_mapping'] = {}

# Map datasets to their appropriate techniques
config['global_settings']['dataset_technique_mapping'].update({
    'custom_classification_test': ['classification_simple'],  # Test dataset
    'custom_classification': ['classification_simple'],        # Full dataset
    'sst2_test': ['classification_simple'],                   # SST-2 test
    'sst2': ['classification_simple'],                        # SST-2 full
    'mnli': ['classification_simple'],                        # MNLI
    'qqp': ['classification_simple'],                         # QQP
})

print(f"\n✅ Updated dataset→technique mappings:")
for dataset, techniques in config['global_settings']['dataset_technique_mapping'].items():
    if dataset in ['custom_classification_test', 'custom_classification', 'sst2_test', 'sst2', 'mnli', 'qqp']:
        print(f"   {dataset} → {techniques}")

# Save
with open('config/prompting_techniques.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print(f"\n✅ Prompting configuration updated! ({len(techniques_to_enable)} techniques enabled)")
print("\n💡 Each dataset now has explicit technique mapping - no more fallback to zero_shot!")

## 2.5 Preview Enabled Prompts

Preview the generated prompt for each enabled experiment before running the benchmark.

For models with `chat_template_mode: hf`, this cell shows both the prompt body and the final model input after Hugging Face `apply_chat_template` is applied.
For manual-template models, the preview is the exact prompt string sent to the model.

In [ ]:
from copy import copy

from pathlib import Path

import inspect

import importlib

import yaml



from experiments import experiment_config as experiment_config_module

from experiments import prompt_manager as prompt_manager_module

from experiments import run_benchmark as run_benchmark_module



importlib.reload(experiment_config_module)

importlib.reload(prompt_manager_module)

importlib.reload(run_benchmark_module)



ExperimentConfig = experiment_config_module.ExperimentConfig

PromptManager = prompt_manager_module.PromptManager

BenchmarkRunner = run_benchmark_module.BenchmarkRunner



MAX_PREVIEWS = 8

MAX_PROMPT_CHARS = 1800

LOAD_HF_TOKENIZERS = True



def _clip(text, limit=MAX_PROMPT_CHARS):

    text = str(text)

    if len(text) <= limit:

        return text

    return text[:limit] + "\n... [truncated]"



def _dataset_length(dataset):

    try:

        return len(dataset)

    except Exception:

        return None



def _dataset_head(dataset, count):

    if count <= 0:

        return []

    if hasattr(dataset, "select"):

        upper = min(count, len(dataset))

        return [dataset[i] for i in range(upper)]

    try:

        return list(dataset[:count])

    except Exception:

        items = []

        iterator = iter(dataset)

        for _ in range(count):

            try:

                items.append(next(iterator))

            except StopIteration:

                break

        return items



def _dataset_skip(dataset, skip_count):

    if skip_count <= 0:

        return dataset

    if hasattr(dataset, "select"):

        return dataset.select(range(skip_count, len(dataset)))

    try:

        return dataset[skip_count:]

    except Exception:

        items = []

        iterator = iter(dataset)

        skipped = 0

        for item in iterator:

            if skipped < skip_count:

                skipped += 1

                continue

            items.append(item)

        return items



def _first_example(dataset):

    if hasattr(dataset, "__getitem__"):

        try:

            return dataset[0]

        except Exception:

            pass

    try:

        iterator = iter(dataset)

        return next(iterator)

    except StopIteration:

        return None



def _extract_input_and_label(example, dataset_config):

    if dataset_config.type == "custom":

        fields = dataset_config.additional_params.get("fields", {})

        text_field = fields.get("text") or fields.get("prompt") or fields.get("question")

        label_field = fields.get("label") or fields.get("answer") or fields.get("reference")

        return example.get(text_field, ""), example.get(label_field, "")



    if dataset_config.type == "huggingface":

        fields = dataset_config.additional_params.get("fields", {})

        text_field = fields.get("text", "text")

        label_field = fields.get("label", "label")

        input_text = example.get(text_field, "")

        true_label = example.get(label_field, "")

        if not input_text:

            input_text = str(example.get("sentence", example.get("question", example.get("text", example))))

        return input_text, true_label



    return str(example.get("text", example.get("question", example))), example.get("label", example.get("answer", ""))



def _normalize_true_label(true_label, dataset_config, dataset_instructions):

    ds_instructions = dataset_instructions.get(dataset_config.name, {})

    label_map = ds_instructions.get("label_map")

    if label_map and true_label in label_map:

        return label_map[true_label]

    if label_map:

        true_key = str(true_label)

        for key, value in label_map.items():

            if str(key) == true_key:

                return value

    return true_label



def _should_use_hf_chat_template(model_config):

    chat_template_mode = str(

        model_config.additional_params.get("chat_template_mode", "manual")

    ).lower()

    return model_config.provider == "huggingface_local" and chat_template_mode in {"hf", "huggingface", "apply_chat_template"}



def _build_local_model_runtime_config(model_config):

    return {

        "name": model_config.name,

        "provider": model_config.provider,

        "model_id": model_config.model_id,

        "max_tokens": model_config.max_tokens,

        "temperature": model_config.temperature,

        "enabled": model_config.enabled,

        **model_config.additional_params,

    }



def _render_prompt_body(prompt_manager, prompt_technique, raw_input, dataset_config, few_shot_examples, use_hf_chat_template):

    if not prompt_technique:

        return raw_input



    if use_hf_chat_template:

        try:

            from utils.generate_prompt_templates import build_inner_prompt, load_configs

            source_cfg = load_configs(Path("./config"))

            dataset_key = prompt_technique.fields.get("dataset")

            technique_key = prompt_technique.fields.get("technique")

            if dataset_key and technique_key:

                template = build_inner_prompt(

                    technique_key=technique_key,

                    dataset_key=dataset_key,

                    technique_templates=source_cfg["technique_templates"],

                    dataset_instructions=source_cfg["dataset_instructions"],

                    few_shot_samples=source_cfg["few_shot_samples"],

                    domain_experts=source_cfg["domain_experts"],

                )

                from string import Template

                return Template(template).safe_substitute({"input": raw_input})

        except Exception as exc:

            print(f"[WARN] Falling back to prompt_manager.apply_technique for {prompt_technique.name}: {exc}")



    apply_sig = inspect.signature(prompt_manager.apply_technique)

    kwargs = {

        "technique": prompt_technique,

        "input_text": raw_input,

        "dataset_name": dataset_config.name,

        "task_type": dataset_config.task_type,

        "examples": few_shot_examples,

    }

    if "output_mode" in apply_sig.parameters:

        kwargs["output_mode"] = "content" if use_hf_chat_template else "manual"

    return prompt_manager.apply_technique(**kwargs)



def _format_with_hf_chat_template(model_config, prompt_body, tokenizer_cache):

    if model_config.name in tokenizer_cache:

        tokenizer = tokenizer_cache[model_config.name]

    else:

        from transformers import AutoTokenizer

        runtime_cfg = _build_local_model_runtime_config(model_config)

        tokenizer = AutoTokenizer.from_pretrained(

            runtime_cfg["model_id"],

            trust_remote_code=runtime_cfg.get("trust_remote_code", False),

        )

        if tokenizer.pad_token is None:

            tokenizer.pad_token = tokenizer.eos_token

        tokenizer_cache[model_config.name] = tokenizer



    if not hasattr(tokenizer, "apply_chat_template"):

        return prompt_body



    try:

        return tokenizer.apply_chat_template(

            [{"role": "user", "content": prompt_body}],

            tokenize=False,

            add_generation_prompt=True,

        )

    except Exception as exc:

        print(f"[WARN] apply_chat_template failed for {model_config.name}: {exc}")

        return prompt_body



config = ExperimentConfig("./config")

config.load_configs()

runner = BenchmarkRunner(

    config=config,

    output_dir="./results",

    experiment_name="prompt_preview",

    enable_prompting=True,

    )



dataset_instructions_path = Path("./config/dataset_instructions.yaml")

with open(dataset_instructions_path, "r", encoding="utf-8") as f:

    dataset_instructions = (yaml.safe_load(f) or {}).get("dataset_instructions", {})



enabled_experiments = config.get_enabled_experiments()

if not enabled_experiments:

    raise ValueError("No enabled experiments found in config/experiments.yaml. Use Mode A or Mode A+ first.")



prompt_manager = runner.prompt_manager or PromptManager()

tokenizer_cache = {}



print(f"Found {len(enabled_experiments)} enabled experiments. Showing up to {MAX_PREVIEWS}.\n")



for exp in enabled_experiments[:MAX_PREVIEWS]:

    model_config = config.get_model_by_name(exp.model)

    dataset_config = config.get_dataset_by_name(exp.dataset)

    prompt_technique = prompt_manager.get_technique_by_name(exp.prompting_technique) if exp.prompting_technique else None



    if not model_config or not dataset_config:

        print(f"[SKIP] {exp.id}: missing model or dataset config")

        continue



    preview_dataset_config = copy(dataset_config)

    if exp.num_samples is not None:

        preview_dataset_config.num_samples = exp.num_samples



    dataset = runner.load_dataset(preview_dataset_config)

    few_shot_examples = None

    preview_dataset = dataset



    if prompt_technique and (

        prompt_technique.technique == "few_shot"

        or "few_shot" in prompt_technique.name.lower()

    ):

        num_examples = prompt_technique.params.get("num_examples", 3)

        dataset_size = _dataset_length(dataset)

        if dataset_size is not None and dataset_size > num_examples:

            few_shot_examples = _dataset_head(dataset, num_examples)

            preview_dataset = _dataset_skip(dataset, num_examples)



    preview_count = _dataset_length(preview_dataset)

    if preview_count == 0:

        print(f"[SKIP] {exp.id}: no preview samples available")

        continue



    example = _first_example(preview_dataset)

    if example is None:

        print(f"[SKIP] {exp.id}: unable to fetch a preview sample")

        continue



    raw_input, true_label = _extract_input_and_label(example, preview_dataset_config)

    true_label = _normalize_true_label(true_label, preview_dataset_config, dataset_instructions)



    use_hf_chat_template = _should_use_hf_chat_template(model_config)

    prompt_body = _render_prompt_body(

        prompt_manager,

        prompt_technique,

        raw_input,

        preview_dataset_config,

        few_shot_examples,

        use_hf_chat_template,

    )



    final_model_input = prompt_body

    if use_hf_chat_template and LOAD_HF_TOKENIZERS and model_config.provider == "huggingface_local":

        final_model_input = _format_with_hf_chat_template(

            model_config,

            prompt_body,

            tokenizer_cache,

        )



    mode_label = model_config.additional_params.get("chat_template_mode", "manual")

    print("=" * 100)

    print(f"Experiment: {exp.id}")

    print(f"Model: {model_config.name} | Dataset: {preview_dataset_config.name} | Technique: {exp.prompting_technique}")

    print(f"Chat template mode: {mode_label}")

    print(f"True label: {true_label}")

    print("-" * 100)



    if use_hf_chat_template:

        print("Prompt body before apply_chat_template:\n")

        print(_clip(prompt_body))

        print("\n" + "-" * 100)

        print("Final model input after apply_chat_template:\n")

        print(_clip(final_model_input))

    else:

        print("Final model input:\n")

        print(_clip(final_model_input))

    print()


## 3. Run Benchmark

This will:
1. Load each enabled model
2. Run predictions on test dataset
3. Show progress bars
4. Save results to `results/` folder

In [ ]:
# Reasoning output toggle for benchmark runs
REASONING_OUTPUT_MODE = True  # Set False to disable reasoning + FINAL_ANSWER format
GEN_PRESET = "reasoning"    # concise | balanced | reasoning | classification

print(f"REASONING_OUTPUT_MODE={REASONING_OUTPUT_MODE}")
print(f"GEN_PRESET={GEN_PRESET}")

In [ ]:
# Run quick test (5 samples)
import subprocess

cmd = ["python", "experiments/run_benchmark.py", "--gen-preset", GEN_PRESET]
if REASONING_OUTPUT_MODE:
    cmd.append("--reasoning-output")

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

## 4. Scale Up (After Test Succeeds)

Once the quick test works, increase sample size for real experiments:

In [ ]:
# Scale up to 100 samples
import yaml

with open('config/datasets.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# STEP 1: Disable ALL datasets first
for dataset in config['datasets']:
    dataset['enabled'] = False

# STEP 2: Enable full datasets with more samples
datasets_to_enable = {
    'sst2': 100,        # Sentiment classification
    'mnli': 100,        # Natural language inference
    # 'qqp': 50,        # Question pairs (uncomment if needed)
}

for dataset in config['datasets']:
    if dataset['name'] in datasets_to_enable:
        dataset['enabled'] = True
        dataset['num_samples'] = datasets_to_enable[dataset['name']]
        print(f"✅ Enabled: {dataset['name']} ({datasets_to_enable[dataset['name']]} samples)")

with open('config/datasets.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print(f"\n✅ Scaled up to full datasets! ({len(datasets_to_enable)} datasets enabled)")
print("💡 Increase sample counts in datasets_to_enable dict for larger experiments")

In [ ]:
# Run full experiment (will take longer)
import subprocess

cmd = ["python", "experiments/run_benchmark.py", "--gen-preset", GEN_PRESET]
if REASONING_OUTPUT_MODE:
    cmd.append("--reasoning-output")

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

## 5. View Results

In [ ]:
# List all result files
!ls -lh results/

In [ ]:
# View latest summary
import json
import glob
from pathlib import Path

# Find latest summary file
summary_files = sorted(glob.glob('results/*_summary.json'))
if summary_files:
    latest = summary_files[-1]
    print(f"📊 Reading: {latest}\n")
    
    with open(latest, 'r', encoding='utf-8') as f:
        results = json.load(f)
    
    print(f"Experiment: {results['experiment_name']}")
    print(f"Total experiments: {len(results['experiments'])}")
    print("\nResults:")
    print("=" * 70)
    
    for exp in results['experiments']:
        if exp['status'] == 'completed':
            model = exp['model']
            dataset = exp['dataset']
            metrics = exp.get('metrics', {})
            accuracy = metrics.get('accuracy', 'N/A')
            print(f"✅ {model:<25} | {dataset:<15} | Accuracy: {accuracy}")
        else:
            print(f"❌ {exp['model']:<25} | {exp['dataset']:<15} | Failed")
else:
    print("No results found. Run the benchmark first!")

In [ ]:
# Create a zip file for easy download
!zip -r results.zip results/
print("\n✅ Results zipped! Download 'results.zip' from the Files panel (left sidebar)")

## 6. Advanced: Try Larger Models

Colab has enough VRAM for 13B models with 4-bit quantization:

In [ ]:
# Enable Llama 2 13B (uses ~7GB VRAM)
import yaml

with open('config/models.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# STEP 1: Disable all models first (optional - keeps existing config)
# Uncomment to start fresh:
# for model in config['models']:
#     model['enabled'] = False

# STEP 2: Enable 13B model (in addition to or instead of current models)
additional_models = [
    'llama-2-13b-4bit',  # High quality 13B model
]

for model in config['models']:
    if model['name'] in additional_models:
        model['enabled'] = True
        print(f"✅ Enabled: {model['name']} (expect ~3-5 min load time)")

with open('config/models.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print("\n⚠️  13B models are slower but produce higher quality results!")
print("💡 Uses ~7GB VRAM - well within Colab T4 limits")

## 💡 Tips for Colab

1. **Free tier limits**: 12-hour sessions, may disconnect if idle
2. **Save often**: Run with small samples first, then scale up
3. **Download results**: Files are deleted when session ends
4. **Monitor GPU**: Run `!nvidia-smi` in a cell to check usage
5. **Reduce samples**: If timeout, reduce `num_samples` in datasets

## 🚀 Next Steps

- Try different prompting techniques (see `config/prompting_techniques.yaml`)
- Add custom datasets (see `data_loaders/data/`)
- Compare multiple models on same dataset
- Export results and analyze locally